In [1]:
import os
import subprocess
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from dotenv import load_dotenv

load_dotenv()

llm = LLM(
    model="openrouter/z-ai/glm-4.5-air:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

@tool("Execute Python Script")
def execute_code(script: str) -> str:
    """Executes a Python script locally in the terminal and returns the output or error trace"""
    try:
        result = subprocess.run(
            ["python", "-c", script],
            capture_output=True,
            text=True,
            timeout=30
        )

        if result.returncode == 0:
            return f"Success! Output:\n{result.stdout}"
        else:
            return f"Error! Error Trace:\n{result.stderr}"

    except Exception as e:
        return f"Error executing script: {str(e)}"

In [2]:
programmer = Agent(
    role="Senior Python Programmer",
    goal="Write clear, efficient, and well-documented Python code to solve the problem given by the prompt.",
    backstory="You are a highly-skilled senior Python programmer with decades of experience developing clear,"
              "memory and space efficient, and easy-to-understand code. You use chain-of-thought reasoning to"
              "break a problem down and then write the relevant code based on your abstraction. Ensure to use"
              "comments to explain your code. Do not invent syntax or libraries that do not exist. Only use"
              "standard Python libraries or well-known third-party libraries and accurate syntax.",
    llm=llm,
    verbose=True
)

tester = Agent(
    role="Senior QA Tester",
    goal="Test code using the execution tool and force fixes if it fails based on the error message.",
    backstory="You are a professional senior QA tester with countless testing, debugging, and fixing code."
              "You ensure the code performs the required function with no logic, syntax, runtime or any"
              "other error. You use the execution tool to run the code and if it fails, you analyze the" 
              "error message, identify the issue, and then instruct the programmer to fix the code. You"
              "repeat this process until the code runs successfully without any errors.",
    tools=[execute_code],
    llm=llm,
    verbose=True
)

In [ ]:
write_code = Task(
    description="{prompt}",
    expected_output="A clean Python script that solves the problem described in the prompt, " 
                    "with clear comments explaining the code.",
    agent=programmer
)

test_code = Task(
    description="Take the code, run it using the tool, and verify it works. If it fails, feed the error"
                "back to the developer to fix. You shouldrepeat this process until the code runs successfully"
                "without any logic, syntax, runtiome or any other type of errors.",
    expected_output="The final working code along with its successful terminal output presented separately.",
    agent=tester,
    context=[write_code]
)

crew = Crew(agents=[programmer, tester], tasks=[write_code, test_code])
user_prompt = input("Enter the programming problem you want to solve: ")

print(crew.kickoff(inputs={"prompt": user_prompt}))

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Programmer                                                                                │
│                                                                                                                 │
│  Task: Given an array of integers nums and an integer target, return the indices i and j such that nums[i] +    │
│  nums[j] == target and i != j.                                                                                  │
│                  You may assume that every input has exactly one pair of indices i and j that satisfy the       │
│  condition.                                                                                                     │
│                  Return the answer with the smaller index first.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Programmer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  def two_sum(nums, target):                                                                                     │
│      """                                                                                                        │
│      Given an array of integers nums and an integer target, return the indices i and j                          │
│      such that nums[i] + nums[j] == target and i != j. You may assume that every input                          │
│      has exactly one pair of indices i and j that satisfy the condition. The answer is                          │
│      returned with the smaller index first.                                                                     │
│                                                                                                                 │
│      Args:                                                                                                      │
│          nums (List[int]): List of integers                                                                     │
│          target (int): Target sum                                                                               │
│                                                                                                                 │
│      Returns:                                                                                                   │
│          List[int]: Indices i and j such that nums[i] + nums[j] == target and i != j                            │
│      """                                                                                                        │
│      # Create a dictionary to store the numbers and their indices                                               │
│      num_dict = {}                                                                                              │
│                                                                                                                 │
│      # Iterate through the array                                                                                │
│      for i, num in enumerate(nums):                                                                             │
│          # Calculate the complement (the number needed to reach the target)                                     │
│          complement = target - num                                                                              │
│                                                                                                                 │
│          # If the complement exists in the dictionary, we've found our pair                                     │
│          if complement in num_dict:                                                                             │
│              # Return the indices with the smaller index first                                                  │
│              # Since we stored the complement's index earlier, it must be smaller than i                        │
│              return [num_dict[complement], i]                                                                   │
│                                                                                                                 │
│          # Store the current number and its index in th

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior QA Tester                                                                                        │
│                                                                                                                 │
│  Task: Take the code, run it using the tool, and verify it works. If it fails, feed the errorback to the        │
│  developer to fix. You shouldrepeat this process until the code runs successfullywithout any logic, syntax,     │
│  runtiome or any other type of errors.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool execute_python_script executed with result: Success! Output:
Test 1: nums=[2, 7, 11, 15], target=9, result=[0, 1]
Test 2: nums=[3, 2, 4], target=6, result=[1, 2]
Test 3: nums=[3, 3], target=6, result=[0, 1]
Test 4: nums=[-1, -2, -3, -4, -5], ta...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior QA Tester                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  def two_sum(nums, target):                                                                                     │
│      """                                                                                                        │
│      Given an array of integers nums and an integer target, return the indices i and j                          │
│      such that nums[i] + nums[j] == target and i != j. You may assume that every input                          │
│      has exactly one pair of indices i and j that satisfy the condition. The answer is                          │
│      returned with the smaller index first.                                                                     │
│                                                                                                                 │
│      Args:                                                                                                      │
│          nums (List[int]): List of integers                                                                     │
│          target (int): Target sum                                                                               │
│                                                                                                                 │
│      Returns:                                                                                                   │
│          List[int]: Indices i and j such that nums[i] + nums[j] == target and i != j                            │
│      """                                                                                                        │
│      # Create a dictionary to store the numbers and their indices                                               │
│      num_dict = {}                                                                                              │
│                                                                                                                 │
│      # Iterate through the array                                                                                │
│      for i, num in enumerate(nums):                                                                             │
│          # Calculate the complement (the number needed to reach the target)                                     │
│          complement = target - num                                                                              │
│                                                                                                                 │
│          # If the complement exists in the dictionary, we've found our pair                                     │
│          if complement in num_dict:                                                                             │
│              # Return the indices with the smaller index first                                                  │
│              # Since we stored the complement's index earlier, it must be smaller than i                        │
│              return [num_dict[complement], i]                                                                   │
│                                                                                                                 │
│          # Store the current number and its index in th

```python
def two_sum(nums, target):
    """
    Given an array of integers nums and an integer target, return the indices i and j 
    such that nums[i] + nums[j] == target and i != j. You may assume that every input 
    has exactly one pair of indices i and j that satisfy the condition. The answer is 
    returned with the smaller index first.
    
    Args:
        nums (List[int]): List of integers
        target (int): Target sum
        
    Returns:
        List[int]: Indices i and j such that nums[i] + nums[j] == target and i != j
    """
    # Create a dictionary to store the numbers and their indices
    num_dict = {}
    
    # Iterate through the array
    for i, num in enumerate(nums):
        # Calculate the complement (the number needed to reach the target)
        complement = target - num
        
        # If the complement exists in the dictionary, we've found our pair
        if complement in num_dict:
            # Return the indices with the smaller index first
 